<a href="https://colab.research.google.com/github/ganesh142007/IRS-01.ipynb/blob/main/irs_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import string
from collections import defaultdict
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Download required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to resolve LookupError

# Folder containing text documents
DOCS_FOLDER = "docs"

# Ensure the directory exists so the script doesn't crash
if not os.path.exists(DOCS_FOLDER):
    os.makedirs(DOCS_FOLDER)
    print(f"Created '{DOCS_FOLDER}' directory. Please add some .txt files to it and re-run.")

# Initialize data structures
inverted_index = defaultdict(set)
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

print("Processing documents...")

# 1. Process each text file in the folder
for filename in os.listdir(DOCS_FOLDER):
    if filename.endswith(".txt"):
        doc_id = filename

        try:
            with open(os.path.join(DOCS_FOLDER, filename), 'r', encoding='utf-8') as file:
                text = file.read().lower()

            # Remove punctuation
            text = text.translate(str.maketrans("", "", string.punctuation))

            # Tokenize words
            tokens = word_tokenize(text)

            # Remove stop words and apply stemming
            for word in tokens:
                if word not in stop_words and word.isalnum():
                    stemmed_word = stemmer.stem(word)
                    # Add the document ID to the set for this specific stem
                    inverted_index[stemmed_word].add(doc_id)

        except Exception as e:
            print(f"Error reading {filename}: {e}")

print(f"Indexing complete! Indexed {len(inverted_index)} unique terms.\n")


# 2. Query Functionality (Search Engine)
def search_index(query):
    """
    Looks up a single-word query or processes multi-word queries using AND logic.
    """
    query_tokens = word_tokenize(query.lower())
    # Clean and stem the query terms just like we did for the documents
    query_stems = [stemmer.stem(w) for w in query_tokens if w not in stop_words and w.isalnum()]

    if not query_stems:
        return "Please enter a valid search term."

    # Initialize results with the document set of the first word
    results = inverted_index.get(query_stems[0], set())

    # Intersect with the document sets of the remaining words (AND logic)
    for stem in query_stems[1:]:
        results = results.intersection(inverted_index.get(stem, set()))

    return results


# --- Test Drive ---
# Example execution (uncomment below if you want to run an interactive loop)
if __name__ == "__main__":
    # Print a tiny sample of the index to verify structure
    print("--- Sample Index Entries ---")
    for token, doc_ids in list(inverted_index.items())[:5]:
        print(f"'{token}': {list(doc_ids)}")
    print("-" * 28 + "\n")

    # Example hardcoded search
    search_term = "python"
    matching_docs = search_index(search_term)
    print(f"Search Results for '{search_term}': {list(matching_docs) if matching_docs else 'No documents found.'}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...


Processing documents...
Indexing complete! Indexed 0 unique terms.

--- Sample Index Entries ---
----------------------------

Search Results for 'python': No documents found.


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
